# GPU Kernel Advisor — Colab backend

FastAPI service next to the T4: accepts a complete CUDA program + kernel name,
compiles with `nvcc`, profiles the named kernel with `ncu`, returns sanitized
metrics JSON plus derived roofline quantities. Exposed publicly via a
cloudflared quick tunnel.

**Run order:** Runtime → T4 GPU, then run all cells top to bottom. The tunnel
cell prints the public URL to paste into the frontend.

**API contract**
- `GET /health` → `{"status": "ok", "gpu": "Tesla T4"}`
- `POST /profile` `{"source": "<complete .cu program with main()>", "kernel_name": "vecAdd"}` →
  - `{"status": "ok", "metrics": {<ncu metric>: {"value", "unit"}}, "derived": {...}}`
  - `{"status": "compile_error", "stderr": "..."}` (frontend shows nvcc output)
  - `{"status": "profile_error", "detail": "...", ...}`

The request must be a *complete* program (with `main()` allocating buffers and
launching the kernel) — the frontend's Monaco editor starts from a template that
includes the harness. `ncu` profiles the first launch matching `kernel_name`.


In [ ]:
%pip install -q fastapi uvicorn


In [ ]:
%%bash
# locate or install ncu (same logic as the spike notebook)
if command -v ncu >/dev/null; then
    echo "ncu on PATH: $(command -v ncu)"
else
    found=$(ls /usr/local/cuda*/bin/ncu /opt/nvidia/nsight-compute/*/ncu 2>/dev/null | head -1 || true)
    if [ -n "$found" ]; then
        ln -sf "$found" /usr/local/bin/ncu
        echo "linked $found"
    else
        CUDA_VER=$(nvcc --version | grep -oP 'release \K[0-9]+\.[0-9]+' | tr . -)
        apt-get -qq update && apt-get -qq install -y cuda-nsight-compute-${CUDA_VER}
        ln -sf /usr/local/cuda/bin/ncu /usr/local/bin/ncu 2>/dev/null || true
    fi
fi
ncu --version | tail -2


In [ ]:
%%writefile server.py
"""FastAPI profiling backend: compile a CUDA program, profile one kernel with ncu,
return sanitized metrics JSON. Runs inside Colab next to the T4."""
import math
import re
import subprocess
import tempfile
import threading
from pathlib import Path

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

ARCH = "sm_75"  # T4
NVCC_TIMEOUT_S = 60
NCU_TIMEOUT_S = 180

METRICS = [
    # timing / launch / clocks
    "gpu__time_duration.sum",
    "launch__registers_per_thread",
    "sm__cycles_elapsed.avg.per_second",           # actual SM clock during the kernel
    # speed-of-light + occupancy
    "sm__throughput.avg.pct_of_peak_sustained_elapsed",
    "dram__throughput.avg.pct_of_peak_sustained_elapsed",
    "sm__warps_active.avg.pct_of_peak_sustained_active",
    # memory system
    "dram__bytes.sum",
    "dram__bytes.sum.per_second",
    "l1tex__t_sector_hit_rate.pct",
    "lts__t_sector_hit_rate.pct",
    # coalescing signal: sectors per global-load request (4 = perfectly coalesced
    # 32-bit accesses, 32 = fully strided)
    "l1tex__average_t_sectors_per_request_pipe_lsu_mem_global_op_ld.ratio",
    # divergence / branches. NOTE: the uniformity pct is vacuous (reads 0) when a
    # kernel compiles to predication and has ~no SASS branches — always interpret
    # it alongside the branch count below, never alone.
    "smsp__thread_inst_executed_per_inst_executed.ratio",
    "smsp__sass_average_branch_targets_threads_uniform.pct",
    "smsp__sass_branch_targets.sum",
    # fp32 work (roofline y-axis; FFMA counts double)
    "smsp__sass_thread_inst_executed_op_fadd_pred_on.sum",
    "smsp__sass_thread_inst_executed_op_fmul_pred_on.sum",
    "smsp__sass_thread_inst_executed_op_ffma_pred_on.sum",
]

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])

# ncu serialises on the GPU perf counters anyway; profile one request at a time
profile_lock = threading.Lock()


class ProfileRequest(BaseModel):
    source: str       # complete .cu program including a main() that launches the kernel
    kernel_name: str  # unmangled base name of the __global__ function to profile


def _num(v):
    try:
        f = float(str(v).replace(",", ""))
        return None if math.isnan(f) else f
    except (TypeError, ValueError):
        return None


def _text(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return None
    return str(v)


def parse_ncu_csv(stdout: str, wanted: list) -> dict:
    """Handle both ncu CSV layouts: long (Metric Name/Unit/Value rows) and
    wide (one column per metric). NaN values are sanitized to None so the
    response is always strict JSON."""
    import io
    import pandas as pd

    lines = stdout.splitlines()
    header_idx = next(
        (i for i, l in enumerate(lines) if l.startswith('"ID"') or l.startswith("ID,")),
        None,
    )
    if header_idx is None:
        raise ValueError("no CSV header in ncu output")
    df = pd.read_csv(io.StringIO("\n".join(lines[header_idx:])), thousands=",")

    if "Metric Name" in df.columns:
        out = {
            row["Metric Name"]: {"value": _num(row["Metric Value"]),
                                 "unit": _text(row.get("Metric Unit"))}
            for _, row in df.iterrows()
        }
    else:
        row = df.iloc[0]
        out = {m: {"value": _num(row[m]), "unit": None} for m in wanted if m in df.columns}

    if not out:
        raise ValueError("CSV parsed but no metrics matched; columns: %s" % list(df.columns))
    return out


def derived_metrics(m: dict) -> dict:
    def g(key):
        return (m.get(key) or {}).get("value") or 0.0

    flops = (g("smsp__sass_thread_inst_executed_op_fadd_pred_on.sum")
             + g("smsp__sass_thread_inst_executed_op_fmul_pred_on.sum")
             + 2.0 * g("smsp__sass_thread_inst_executed_op_ffma_pred_on.sum"))
    dram_bytes = g("dram__bytes.sum")
    dur_ns = g("gpu__time_duration.sum")
    return {
        "fp32_flops": flops,
        "arithmetic_intensity_flop_per_byte": flops / dram_bytes if dram_bytes else None,
        "achieved_gflops": flops / dur_ns if dur_ns else None,          # FLOP/ns == GFLOP/s
        "achieved_dram_gbps": dram_bytes / dur_ns if dur_ns else None,  # B/ns == GB/s
        "duration_ms": dur_ns / 1e6 if dur_ns else None,
    }


@app.get("/health")
def health():
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip()
    return {"status": "ok", "gpu": gpu}


@app.post("/profile")
def profile(req: ProfileRequest):
    with profile_lock, tempfile.TemporaryDirectory() as td:
        src = Path(td) / "kernel.cu"
        binary = Path(td) / "kernel"
        src.write_text(req.source)

        try:
            compile_res = subprocess.run(
                ["nvcc", "-O3", "-arch=" + ARCH, "-lineinfo",
                 "-o", str(binary), str(src)],
                capture_output=True, text=True, timeout=NVCC_TIMEOUT_S,
            )
        except subprocess.TimeoutExpired:
            return {"status": "compile_error", "stderr": "nvcc timed out"}
        if compile_res.returncode != 0:
            return {"status": "compile_error", "stderr": compile_res.stderr[-4000:]}

        try:
            ncu_res = subprocess.run(
                ["ncu", "--csv",
                 "--kernel-name", "regex:" + re.escape(req.kernel_name),
                 "--launch-count", "1",
                 "--metrics", ",".join(METRICS),
                 str(binary)],
                capture_output=True, text=True, timeout=NCU_TIMEOUT_S, cwd=td,
            )
        except subprocess.TimeoutExpired:
            return {"status": "profile_error",
                    "detail": "ncu timed out after %ds" % NCU_TIMEOUT_S}

        combined = ncu_res.stdout + ncu_res.stderr
        if "ERR_NVGPUCTRPERM" in combined:
            return {"status": "profile_error",
                    "detail": "GPU performance counters blocked (ERR_NVGPUCTRPERM)"}

        try:
            metrics = parse_ncu_csv(ncu_res.stdout, METRICS)
        except ValueError as e:
            return {
                "status": "profile_error",
                "detail": str(e),
                "ncu_stdout_tail": ncu_res.stdout[-2000:],
                "ncu_stderr_tail": ncu_res.stderr[-2000:],
            }

        return {"status": "ok", "metrics": metrics, "derived": derived_metrics(metrics)}


In [ ]:
import threading
import time

import requests
import uvicorn

import server  # imports server.py — fails fast here if it has a syntax error

config = uvicorn.Config("server:app", host="127.0.0.1", port=8000, log_level="warning")
uv_server = uvicorn.Server(config)
threading.Thread(target=uv_server.run, daemon=True).start()

for _ in range(20):
    try:
        print("health:", requests.get("http://127.0.0.1:8000/health", timeout=2).json())
        break
    except Exception:
        time.sleep(0.5)
else:
    raise RuntimeError("server did not come up")


In [ ]:
import pathlib
import re
import subprocess
import time

bin_path = "/usr/local/bin/cloudflared"
if not pathlib.Path(bin_path).exists():
    subprocess.run(
        ["wget", "-q",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
         "-O", bin_path], check=True)
    subprocess.run(["chmod", "+x", bin_path], check=True)

proc = subprocess.Popen(
    [bin_path, "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url = None
deadline = time.time() + 30
while time.time() < deadline and url is None:
    line = proc.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
    if m:
        url = m.group(0)

if url:
    print("PUBLIC BACKEND URL:", url)
    print("(paste this into the frontend; it changes every time this cell reruns)")
else:
    proc.kill()
    print("no tunnel URL within 30s — rerun this cell (quick tunnels are flaky to start)")


In [ ]:
import json

import requests

VECTOR_ADD = r'''
#include <cstdio>
#include <cuda_runtime.h>

#define CUDA_CHECK(call)                                                      \
    do {                                                                      \
        cudaError_t err_ = (call);                                            \
        if (err_ != cudaSuccess) {                                            \
            fprintf(stderr, "CUDA error: %s at %s:%d\n",                      \
                    cudaGetErrorString(err_), __FILE__, __LINE__);            \
            return 1;                                                         \
        }                                                                     \
    } while (0)

__global__ void vecAdd(const float* __restrict__ a,
                       const float* __restrict__ b,
                       float* __restrict__ c, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}

int main()
{
    const int n = 1 << 24;
    const size_t bytes = (size_t)n * sizeof(float);
    float *a, *b, *c;
    CUDA_CHECK(cudaMalloc(&a, bytes));
    CUDA_CHECK(cudaMalloc(&b, bytes));
    CUDA_CHECK(cudaMalloc(&c, bytes));
    float* h = (float*)malloc(bytes);
    for (int i = 0; i < n; ++i) h[i] = 1.0f;
    CUDA_CHECK(cudaMemcpy(a, h, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(b, h, bytes, cudaMemcpyHostToDevice));
    vecAdd<<<(n + 255) / 256, 256>>>(a, b, c, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    printf("done\n");
    return 0;
}
'''

# happy path: full profile round-trip through the local server
r = requests.post("http://127.0.0.1:8000/profile",
                  json={"source": VECTOR_ADD, "kernel_name": "vecAdd"},
                  timeout=300)
resp = r.json()
print(json.dumps(resp, indent=2))
assert resp["status"] == "ok", "profile round-trip failed"
assert resp["derived"]["arithmetic_intensity_flop_per_byte"] is not None

# compile-error path: frontend needs structured errors, not a 500
bad = requests.post("http://127.0.0.1:8000/profile",
                    json={"source": "__global__ void broken( {", "kernel_name": "broken"},
                    timeout=60)
print("compile-error path:", bad.json()["status"])
assert bad.json()["status"] == "compile_error"
print("\nALL BACKEND TESTS PASSED")


## Operational notes

- **The tunnel URL changes on every rerun** (quick tunnels are ephemeral), so the
  frontend must have a user-editable "backend URL" field — never hardcode it.
- **Keep this tab open.** Free-tier Colab reclaims idle runtimes; after a reset,
  Run All again and paste the new URL.
- **Latency:** each profile request is nvcc (~2–5 s) + ncu with replay passes
  (~3–10 s). Size the frontend's loading state accordingly.
- **Security:** this endpoint compiles and runs arbitrary code on your Colab
  runtime. That's the product, and it's your own throwaway VM — but don't post
  the tunnel URL anywhere public while it's live.
- **Parser hardening (later):** if the CSV layout causes trouble across ncu
  versions, switch to `ncu --export` + the `ncu_report` Python module shipped in
  Nsight Compute's `extras/python` — reads the binary report, no CSV scraping.
